# PyEncode — Quantum Amplitude Encoding of FEM Load Vectors

**Author:** Krishnan Suresh · University of Wisconsin–Madison

---

PyEncode is a Python compiler that takes ordinary NumPy code constructing
a structured load vector **f** and returns an efficient Qiskit circuit
that prepares the corresponding quantum state

$$|\psi_f\rangle = \frac{1}{\|\mathbf{f}\|} \sum_{k=0}^{N-1} f_k\,|k\rangle.$$

For each recognised pattern, PyEncode synthesises a *dedicated* circuit
whose gate count scales significantly better than the general
Möttönen / Shende algorithm (O(2ᵐ) gates).

This notebook demonstrates every supported load type for **N = 64** nodes
(m = 6 qubits), shows the resulting circuits, and compares gate counts
against the Shende baseline.

> **PNG figures** are written to `figures/` whenever this notebook is
> executed from top to bottom (Run All).


## 1 · Imports and configuration

In [1]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display
from qiskit.circuit.library import StatePreparation
from qiskit import QuantumCircuit, transpile
from pyencode import encode

# ── Global settings ────────────────────────────────────────────────────────

BASIS = ['cx', 'u', 'x', 'h', 'ry', 'rz', 'rx', 'p']   # transpilation basis




## 2 · Helper functions

Defined once; reused by every load-type section below.

In [2]:


# ── Circuit figure ─────────────────────────────────────────────────────────

def show_circuit(code_str, filename = None, decompose_reps=0, fallback_vector=None):
    """
    Encode *code_str*, draw the resulting circuit inline and save to FIG_DIR.
    Returns (circuit, info) for further inspection.
    """
    circuit, info = encode(code_str, fallback_vector=fallback_vector)
    qc = circuit.decompose(reps=decompose_reps) if decompose_reps > 0 else circuit

    fig = qc.draw(
        "mpl",
        style="iqp",
        fold=-1,
        plot_barriers=False,
        scale=0.5,
    )
    path = None

    display(fig)
    plt.close(fig)
    print(f"  → saved {path}")
    return circuit, info


# ── Shende gate count (for comparison) ────────────────────────────────────

def shende_gates(fv):
    """Return the transpiled gate count for Shende's general StatePreparation."""
    fv = np.array(fv, dtype=complex)
    fv /= np.linalg.norm(fv)
    sp = StatePreparation(fv)
    m = np.log2(len(fv)).astype(int)
    qc = QuantumCircuit(m)
    qc.append(sp, range(m))
    qc = qc.decompose(reps=10)
    t = transpile(qc, basis_gates=BASIS, optimization_level=0)
    return sum(t.count_ops().values())


print("Helper functions defined.")


Helper functions defined.


In [5]:
N     = 32          
m     = np.log2(N).astype(int) 
x = np.linspace(0, 1.0, N)   # spatial coordinate array used throughout

example1 = f"""
N = {N}
b = np.zeros(N)
b[6] = 1.0
"""

circuit, info = encode(example1, fallback_vector=None)
print(circuit)
print(info)
print(info.circuit_code)



          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     ├───┤
q_2: ┤ X ├
     └───┘
q_3: ─────
          
q_4: ─────
          
PyEncode  v0.1.0
  Load type   : POINT_LOAD
  N           : 32  (m = 5 qubits)
  Gate count  : 2
  Complexity  : O(m)
  Fallback    : no
  Parameters  : {'k': 6, 'P': 1.0}
# PyEncode — emitted circuit: POINT_LOAD  k=6
# m = 5 qubits,  N = 32 nodes
# Edit freely; run as standalone Qiskit code.

from qiskit import QuantumCircuit

qc = QuantumCircuit(5, name='point_load')

# Encode index k=6 = 0b00110 in binary (LSB = qubit 0).
# Apply X on each qubit where the corresponding bit of k is 1.
qc.x(1)  # bit 1 of 6 is 1
qc.x(2)  # bit 2 of 6 is 1

# Circuit prepares |psi> = |k>
